# Sprint 7B - GATv2 Topology/Context Runner

**Runner-only notebook.** Model, graph loading, training, evaluation, plotting, and reporting logic stays in the repository under `src/`, `scripts/`, and `configs/`.

Execution plan: `docs/exec-plans/active/007b-sprint7b-gatv2-topology-attention.md`  
Runner boundary: `colab/README.md`

Before starting, confirm that the approved code revision is pushed and Drive contains the raw data plus Sprint 3/Sprint 5B processed graph artifacts.

## Step 1 - Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2 - Clone Or Update Repo Checkout

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/YasinEkici/crispr-gnn-offtarget.git"
REPO_DIR="/content/crispr-gnn-offtarget"
GIT_REF="sprint7/gat-gatv2"
if [ -d "$REPO_DIR/.git" ]; then
  cd "$REPO_DIR"
  git fetch origin "$GIT_REF"
  git checkout "$GIT_REF"
  git pull --ff-only origin "$GIT_REF"
else
  git clone --branch "$GIT_REF" "$REPO_URL" "$REPO_DIR"
  cd "$REPO_DIR"
fi
git rev-parse --short HEAD


## Step 3 - Dependency Sync And Runtime Check

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
python -m pip install -q uv
uv sync
uv run python - <<'PY'
import torch
import torch_geometric
print('torch', torch.__version__)
print('torch_geometric', torch_geometric.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
PY


## Step 4 - Copy Drive Data And Processed Graph Artifacts

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
DRIVE_ROOT_CANDIDATES=(
  "${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
  "/content/drive/MyDrive/crispr-gnn-offtarget"
)
DRIVE_ROOT=""
for CANDIDATE in "${DRIVE_ROOT_CANDIDATES[@]}"; do
  if [ -d "$CANDIDATE" ]; then
    DRIVE_ROOT="$CANDIDATE"
    break
  fi
done
if [ -z "$DRIVE_ROOT" ]; then
  echo "No Drive project root found. Checked:" >&2
  printf '  %s\n' "${DRIVE_ROOT_CANDIDATES[@]}" >&2
  echo "Available MyDrive directories:" >&2
  find /content/drive/MyDrive -maxdepth 1 -type d | sort >&2
  exit 1
fi
echo "Using DRIVE_ROOT=$DRIVE_ROOT"
mkdir -p data/raw data/processed
if [ -d "$DRIVE_ROOT/data/raw" ]; then
  rsync -a "$DRIVE_ROOT/data/raw/" data/raw/
else
  echo "Missing $DRIVE_ROOT/data/raw; raw data is required to build Sprint 7B artifacts" >&2
  exit 1
fi
if [ -d "$DRIVE_ROOT/data/processed" ]; then
  rsync -a "$DRIVE_ROOT/data/processed/" data/processed/
else
  echo "Missing $DRIVE_ROOT/data/processed; Sprint 3 graph artifacts are required" >&2
  exit 1
fi
if [ ! -d data/processed/graphs/sprint3/graph_b_guide_similarity_control ]; then
  echo "Missing local Sprint 3 Graph B artifacts after copy: data/processed/graphs/sprint3/graph_b_guide_similarity_control" >&2
  find data/processed/graphs -maxdepth 3 -type d | sort >&2 || true
  exit 1
fi
if [ ! -d data/processed/graphs/sprint5b/graph_c_context_observation ]; then
  echo "Sprint 5B Graph C artifact not found locally; Step 5 will rebuild it from Sprint 3 + raw data."
fi
find data/processed/graphs -maxdepth 3 -type f -name 'manifest.json' | sort


## Step 5 - Build And Validate Sprint 7B Graph B S5F2 Artifact

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
if [ ! -d data/processed/graphs/sprint5b/graph_c_context_observation ] || [ ! -f data/processed/graphs/sprint5b/graph_c_context_observation/features_S5F2_energy.parquet ]; then
  echo "Building Sprint 5B Graph C S5F2 artifact required by Sprint 7B..."
  uv run python scripts/build_sprint5b_graph_c_energy_features.py \
    --data-config configs/data/mak2022.yaml \
    --schema-config configs/sweeps/graph_schema_ablation.yaml \
    --source-artifact-dir data/processed/graphs/sprint3 \
    --artifact-dir data/processed/graphs/sprint5b \
    --report-path outputs/sprint5b/graph_c_energy_sensitivity_artifact_report.md
fi
uv run python scripts/build_sprint7b_graph_b_energy_features.py \
  --data-config configs/data/mak2022.yaml \
  --schema-config configs/sweeps/graph_schema_ablation.yaml \
  --source-artifact-dir data/processed/graphs/sprint3 \
  --artifact-dir data/processed/graphs/sprint7b \
  --report-path outputs/sprint7b/graph_b_s5f2_artifact_report.md
PYTHONPATH=src uv run python - <<'PY'
from pathlib import Path
from crispr_gnn.graph.graph_schemas import GRAPH_B, GRAPH_C
from crispr_gnn.graph.pyg_dataset import Sprint3HeteroDataLoader
graph_b = Sprint3HeteroDataLoader(Path('data/processed/graphs/sprint7b')).load(GRAPH_B)
graph_c = Sprint3HeteroDataLoader(Path('data/processed/graphs/sprint5b')).load(GRAPH_C)
assert graph_b.manifest['feature_tables']['S5F2_energy'] == 268
assert graph_c.manifest['feature_tables']['S5F2_energy'] == 268
assert graph_c.manifest['feature_tables']['target_observation_features'] > 0
print('Graph B features:', graph_b.manifest['feature_tables'])
print('Graph C features:', graph_c.manifest['feature_tables'])
PY


## Step 6 - Run Sprint 7B GATv2 Topology Comparison

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
RUN_ID="sprint7b_gatv2_topology_seed42_$(date -u +%Y%m%d_%H%M%S)"
uv run python scripts/run_sprint7b_gatv2_topology.py \
  --config configs/sweeps/sprint7b_gatv2_topology.yaml \
  --run-id "$RUN_ID"
echo "$RUN_ID" > /content/sprint7b_gatv2_topology_run_id.txt


## Step 7 - Copy Outputs Back To Drive

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
DRIVE_ROOT="${DRIVE_ROOT:-/content/drive/MyDrive/crispr_gnn_offtarget}"
ALT_DRIVE_ROOT="/content/drive/MyDrive/crispr-gnn-offtarget"
if [ ! -d "$DRIVE_ROOT" ] && [ -d "$ALT_DRIVE_ROOT" ]; then
  DRIVE_ROOT="$ALT_DRIVE_ROOT"
fi
RUN_BASENAME=$(cat /content/sprint7b_gatv2_topology_run_id.txt)
LOCAL_OUT="outputs/sprint7b"
RETURN_ROOT="$DRIVE_ROOT/returned_outputs/$RUN_BASENAME"
if [ -e "$RETURN_ROOT" ]; then
  echo "Output already exists in Drive: $RETURN_ROOT" >&2
  exit 1
fi
mkdir -p "$RETURN_ROOT"
rsync -a "$LOCAL_OUT/" "$RETURN_ROOT/"
find "$RETURN_ROOT" -maxdepth 3 -type f | sort | head -100


## Step 8 - Returned Artifact Checks

In [ ]:
%%bash
set -euo pipefail
cd /content/crispr-gnn-offtarget
OUT="outputs/sprint7b"
test -f "$OUT/gatv2_topology_comparison.csv"
test -f "$OUT/gatv2_topology_report.md"
test -f "$OUT/gatv2_topology_run_manifest.json"
test -f "$OUT/graph_artifact_provenance.json"
test -f "$OUT/graph_b_s5f2_artifact_report.md"
test -d "$OUT/diagnostics"
test -d "$OUT/figures"
test -f "$OUT/diagnostics/gatv2_topology_attention_summary.csv"
test -f "$OUT/figures/gatv2_topology_auprc_comparison.png"
test -f "$OUT/figures/gatv2_topology_attention_by_edge_kind.png"
PYTHONPATH=src uv run python - <<'PY'
import json
from pathlib import Path
import pandas as pd
manifest = json.loads(Path('outputs/sprint7b/gatv2_topology_run_manifest.json').read_text())
ids = {run['predeclared_id'] for run in manifest['runs']}
expected = {
    'S7B_REF_XGB_F4', 'S7B_REF_GA_GCN', 'S7B_REF_GA_GATV2', 'S7B_REF_GC_GCN',
    'S7B_R1_graph_b_gcn_s5f2', 'S7B_R2_graph_b_gatv2_s5f2', 'S7B_R3_graph_c_gatv2_s5f2',
}
if ids != expected:
    raise SystemExit(f'Unexpected Sprint 7B run IDs: {ids}')
results = pd.read_csv('outputs/sprint7b/gatv2_topology_comparison.csv')
print(results[['predeclared_run_id','graph_schema','architecture','test_auprc','test_mcc','test_specificity','test_tn','test_fp']].to_string(index=False))
print('validated', manifest['batch_id'])
PY
